# **Import the needed Modules and Libraries**

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import pandas as pd
from selenium.webdriver.support.ui import WebDriverWait # report 
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
import time

# **Download and Use the correct ChromeDriver**

In [2]:
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)
driver.get("https://www.google.com")
serch_box = driver.find_element(By.XPATH, "//textarea[@id='APjFqb']")

In [ ]:
#Get Noon website first page
driver.get("https://www.noon.com/egypt-ar/electronics-and-mobiles/mobiles-and-accessories/mobiles-20905/?page=1")

In [ ]:
# Get the full HTML of the first product card to know the structer of the data and how to extract it 
cards = driver.find_elements(By.CSS_SELECTOR, "div[class*='layoutWrapper'] > div")
print(cards[0].get_attribute("innerHTML"))

<a class="PBoxLinkHandler-module-scss-module__WvRpgq__productBoxLink" href="/egypt-ar/smart-10-dual-sim-iris-blue-3-3gb-ram-64gb-4g-middle-east-version/N70182485V/p/?o=dd48f512f7640a9d"><div class="ProductBoxVertical-module-scss-module__NG8XsG__wrapper"><div data-qa="productImagePLP_سمارت 10 ثنائي الشريحة 3+3 جيجابايت + 64 جيجابايت أيريس أزرق - النسخة الشرق أوسطية" class="ProductImageSection-module-scss-module__8NuU7G__imageSection"><div class="ProductImageCarousel-module-scss-module__SlkSTG__wrapper" style="--transition-duration:1000ms"><div class="ProductImageCarousel-module-scss-module__SlkSTG__slidesWrapper"><div class="ProductImageCarousel-module-scss-module__SlkSTG__slide"><div class="ProductImage-module-scss-module__DS-hBG__imageContainer"><div class="ProductImage-module-scss-module__DS-hBG__imageWrapper"><img class="ProductImageCarousel-module-scss-module__SlkSTG__productImage" width="auto" height="100%" src="https://f.nooncdn.com/p/pnsku/N70182485V/45/_/1764242115/8defba4c-639

# **Start Scraping and Exctrating the needed data**
(Product_Name, Product_Price, Product_Rating, Produt_Reviews, Product_Photo_URL)

In [8]:
get_names = []
get_prices = []
get_ratings =[]
get_reviews =[]
get_photo_URLs =[]

for page in range(1,19):
    print(f"Scraping page {page}/18")
    url = f"https://www.noon.com/egypt-ar/electronics-and-mobiles/mobiles-and-accessories/mobiles-20905/?page={page}"
    driver.get(url)
    time.sleep(4)

    #Get product_name
    name_elements = driver.find_elements(
        By.CSS_SELECTOR,
        "div.ProductDetailsSection-module-scss-module__Y6u1Qq__wrapper > h2")
    names =[]
    for element in name_elements:
        names.append(element.text.strip())
    
    #Get product_price
    price_elements = driver.find_elements(
        By.CSS_SELECTOR,
        "strong.Price-module-scss-module__q-4KEG__amount"
    )
    price = []
    for element in price_elements:
        price.append(element.text.strip())
    
    #Get product_rating
    rating_elements = driver.find_elements(
        By.CSS_SELECTOR,
        "div.RatingPreviewStar-module-scss-module__zCpaOG__textCtr"
    )
    rating = []
    for element in rating_elements:
        rating.append(element.text.strip())

    #Get product_reviews
    review_elements = driver.find_elements(
        By.CSS_SELECTOR,
        "div.RatingPreviewStar-module-scss-module__zCpaOG__countCtr span"
    )
    reviews = []
    for el in review_elements:
        reviews.append(el.text.strip())

    #Get Photos_Url
    photo_elements = driver.find_elements(
        By.CSS_SELECTOR,
        "img.ProductImageCarousel-module-scss-module__SlkSTG__productImage"
    )
    photos = []
    for el in photo_elements:
        photos.append(el.get_attribute("src"))
    

    #alignment all the product features to name len to create df
    # ── Align all lists to names length ────────────────────────
    n = len(names)

    while len(price) < n:
        price.append("N/A")
    price = price[:n]

    while len(rating) < n:
        rating.append("N/A")
    rating = rating[:n]

    while len(reviews) < n:
        reviews.append("N/A")
    reviews = reviews[:n]

    while len(photos) < n:
        photos.append("N/A")
    photos = photos[:n]

    #append in master lst
    for name in names:
        get_names.append(name)

    for price in price:
        get_prices.append(price)

    for rating in rating:
        get_ratings.append(rating)

    for review in reviews:
        get_reviews.append(review)

    for photo in photos:
        get_photo_URLs.append(photo)

    print(f"  Page {page} done: {n} products found | Total so far: {len(get_names)}")
    time.sleep(2)

print(f"\n Total products scraped: {len(get_names)}")


Scraping page 1/18
  Page 1 done: 50 products found | Total so far: 50
Scraping page 2/18
  Page 2 done: 50 products found | Total so far: 100
Scraping page 3/18
  Page 3 done: 50 products found | Total so far: 150
Scraping page 4/18
  Page 4 done: 50 products found | Total so far: 200
Scraping page 5/18
  Page 5 done: 50 products found | Total so far: 250
Scraping page 6/18
  Page 6 done: 48 products found | Total so far: 298
Scraping page 7/18
  Page 7 done: 50 products found | Total so far: 348
Scraping page 8/18
  Page 8 done: 50 products found | Total so far: 398
Scraping page 9/18
  Page 9 done: 49 products found | Total so far: 447
Scraping page 10/18
  Page 10 done: 49 products found | Total so far: 496
Scraping page 11/18
  Page 11 done: 49 products found | Total so far: 545
Scraping page 12/18
  Page 12 done: 50 products found | Total so far: 595
Scraping page 13/18
  Page 13 done: 50 products found | Total so far: 645
Scraping page 14/18
  Page 14 done: 50 products found | T

In [12]:
#Store extracted data in dataframe
data = pd.DataFrame({
    "Product Name": get_names,
    "Price": get_prices,
    "Rating": get_ratings,
    "Reviews": get_reviews,
    "Photo URL": get_photo_URLs
})

data.head(10)

,Product Name,Price,Rating,Reviews,Photo URL
0,إنفينيكس سمارت 10 ثنائي الشريحة 3+3 جيجابايت +...,"5,099",4.4,724,https://f.nooncdn.com/p/pnsku/N70182485V/45/_/...
1,سامسونج جالاكسي A17 ثنائي الشريحة 4G أسود 6GB ...,"9,999",4.5,1.2K,https://f.nooncdn.com/p/pzsku/Z31ECA424BE18ABD...
2,إنفينيكس سمارت 10 ثنائي الشريحة 3+3 جيجابايت +...,"5,199",4.4,724,https://f.nooncdn.com/p/pnsku/N70182486V/45/_/...
3,سامسونج جالاكسي A17 ثنائي الشريحة 4G أسود 8GB ...,"11,999",4.5,1.2K,https://f.nooncdn.com/p/pzsku/ZC23004F18866BE5...
4,سامسونج جالاكسي A07 ثنائي الشريحة أسود 4GB 128...,"7,240",4.5,2.0K,https://f.nooncdn.com/p/pzsku/Z549671868916F4F...
5,إنفينيكس هوت 60 برو ثنائي الشريحة أنيق أسود 8+...,"9,700",4.6,166,https://f.nooncdn.com/p/pnsku/N70204530V/45/_/...
6,إيتيل A90 ثنائي الشريحة أسود متلألئ 3+5 جيجابا...,"4,149",4.2,2.2K,https://f.nooncdn.com/p/pnsku/N70169194V/45/_/...
7,إنفينيكس سمارت 10 ثنائي الشريحة 4+4 جيجابايت +...,"6,342",4.4,1.3K,https://f.nooncdn.com/p/pnsku/N70182484V/45/_/...
8,فيفو V50 لايت AI 5G ثنائي الشريحة ذهب تيتانيوم...,"14,999",4.3,1.3K,https://f.nooncdn.com/p/pnsku/N70166558V/45/_/...
9,سامسونج جالاكسي A07 ثنائي الشريحة لون بنفسجي ف...,"9,845",5.0,9,https://f.nooncdn.com/p/pzsku/ZBD74DD5FAFD27D6...


In [ ]:
#Data Cleaning 
data = data.drop_duplicates(subset="Product Name")
data["Price"] = data["Price"].apply(lambda x: float(x.replace(",", "")) if x != "N/A" else None)
data = data.reset_index(drop=True)

print(f"Final dataset: {data.shape[0]} products, {data.shape[1]} columns")
data.head(10)

Final dataset: 860 products, 5 columns


,Product Name,Price,Rating,Reviews,Photo URL
0,إنفينيكس سمارت 10 ثنائي الشريحة 3+3 جيجابايت +...,5099.0,4.4,724,https://f.nooncdn.com/p/pnsku/N70182485V/45/_/...
1,سامسونج جالاكسي A17 ثنائي الشريحة 4G أسود 6GB ...,9999.0,4.5,1.2K,https://f.nooncdn.com/p/pzsku/Z31ECA424BE18ABD...
2,إنفينيكس سمارت 10 ثنائي الشريحة 3+3 جيجابايت +...,5199.0,4.4,724,https://f.nooncdn.com/p/pnsku/N70182486V/45/_/...
3,سامسونج جالاكسي A17 ثنائي الشريحة 4G أسود 8GB ...,11999.0,4.5,1.2K,https://f.nooncdn.com/p/pzsku/ZC23004F18866BE5...
4,سامسونج جالاكسي A07 ثنائي الشريحة أسود 4GB 128...,7240.0,4.5,2.0K,https://f.nooncdn.com/p/pzsku/Z549671868916F4F...
5,إنفينيكس هوت 60 برو ثنائي الشريحة أنيق أسود 8+...,9700.0,4.6,166,https://f.nooncdn.com/p/pnsku/N70204530V/45/_/...
6,إيتيل A90 ثنائي الشريحة أسود متلألئ 3+5 جيجابا...,4149.0,4.2,2.2K,https://f.nooncdn.com/p/pnsku/N70169194V/45/_/...
7,إنفينيكس سمارت 10 ثنائي الشريحة 4+4 جيجابايت +...,6342.0,4.4,1.3K,https://f.nooncdn.com/p/pnsku/N70182484V/45/_/...
8,فيفو V50 لايت AI 5G ثنائي الشريحة ذهب تيتانيوم...,14999.0,4.3,1.3K,https://f.nooncdn.com/p/pnsku/N70166558V/45/_/...
9,سامسونج جالاكسي A07 ثنائي الشريحة لون بنفسجي ف...,9845.0,5.0,9,https://f.nooncdn.com/p/pzsku/ZBD74DD5FAFD27D6...


In [15]:
data.isnull().sum()

Product Name    0
Price           0
Rating          0
Reviews         0
Photo URL       0
dtype: int64

In [16]:
data.to_csv("noon_products.csv", index=False)